# Phase 1 — OCR

**What this notebook does:**
1. Mounts Google Drive
2. Reads `data_information.csv` to get the list of documents to process
3. For each PDF, converts page-by-page at 300 DPI and runs MonkeyOCR
4. Saves OCR output under `output/{classification}/{document_group_id}/{filename}/page_{N}/`
5. Records progress to `checkpoint/ocr_checkpoint.json` — already-done pages are skipped on re-run

**Drive layout expected:**
```
RAG_OCRv1/
├── MonkeyOCR/               ← MonkeyOCR repo
├── data/
│   ├── data_information.csv
│   └── {document_group_id}/{filename}.pdf
├── checkpoint/
│   └── ocr_checkpoint.json  ← written by this notebook
└── output/
    └── {classification}/{document_group_id}/{filename}/page_{N}/
        ├── page.png         ← rasterised page image
        └── <monkeyocr outputs: *.md, *.json, images…>
```

**Next notebook:** `phase1_llm_parsing.ipynb` — reads the MonkeyOCR markdown output and runs LLM table normalisation + markdown cleaning.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/mydrive', force_remount=True)

Mounted at /content/mydrive


## 2. Install dependencies

In [2]:
!pip install transformers==4.57.6 \
    lmdeploy==0.9.2 \
    pypdfium2==5.6.0 \
    pdf2image==1.17.0 \
    boto3 \
    fitz \
    PyMuPDF==1.24.14 \
    loguru \
    pdfminer \
    pdfminer.six \
    pandas==2.2.2 \
    protobuf==5.29.6 \
    pillow==11.3.0 \
    psutil==5.9.5 \
    fast_langdetect \
    qwen-vl-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 42.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 133.4 MB/s eta 0:00:00
   ━━━━━━

In [1]:
# PaddlePaddle GPU + PaddleX
!pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu130/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu130/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB ? eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 46.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 47.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 MB ? eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 420.9/420.9 MB 767.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.9/145.9 MB 7.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170

In [2]:
!pip install paddlex

  Using cached paddlex-3.4.3-py3-none-any.whl.metadata (80 kB)
  Using cached aistudio_sdk-0.3.8-py3-none-any.whl.metadata (1.1 kB)
  Using cached modelscope-1.35.3-py3-none-any.whl.metadata (43 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.7/415.7 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.0/80.0 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: PyYAML
    Found existing installation: PyYAML 6.0.3
    Uninstalling PyYAML-6.0.3:
      Successfully uninstalled PyYAML-6.0.3


In [3]:
!pip uninstall -y torch torchvision torchaudio triton
!pip install torch==2.10.0+cu130 torchvision==0.25.0+cu130 torchaudio==2.10.0+cu130 --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torch 2.7.1
Uninstalling torch-2.7.1:
  Successfully uninstalled torch-2.7.1
Found existing installation: torchvision 0.22.1
Uninstalling torchvision-0.22.1:
  Successfully uninstalled torchvision-0.22.1
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Found existing installation: triton 3.3.1
Uninstalling triton-3.3.1:
  Successfully uninstalled triton-3.3.1
Looking in indexes: https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 85.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 351.3/351.3 MB 67.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 55.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 114.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 81.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.

## 3. Configuration — edit these paths if needed

In [4]:
from pathlib import Path

# ── Drive root ────────────────────────────────────────────────────────────────
DRIVE_BASE = Path("/content/mydrive/MyDrive/RAG_OCRv1")

# ── Input ─────────────────────────────────────────────────────────────────────
# CSV   : DRIVE_BASE/data/data_information.csv
# PDFs  : DRIVE_BASE/data/{document_group_id}/{filename}
DATA_DIR = DRIVE_BASE / "data"
CSV_PATH = DATA_DIR   / "data_information.csv"

# ── Output (stays on Drive so it persists across sessions) ───────────────────
# DRIVE_BASE/output/{classification}/{document_group_id}/{filename_stem}/page_{N}/
OUTPUT_DIR = DRIVE_BASE / "output"

# ── Checkpoint (stays on Drive) ───────────────────────────────────────────────
CHECKPOINT_DIR = DRIVE_BASE / "checkpoint"
OCR_CHECKPOINT = CHECKPOINT_DIR / "ocr_checkpoint.json"

# ── Processing options ────────────────────────────────────────────────────────
DPI                   = 300   # rasterisation resolution
LIMIT                 = None  # int -> process only N document groups; None = all
DOC_GROUP_FILTER      = None  # e.g. "panasonic_aircon_CS-PW24KE" to test one group
CLASSIFICATION_FILTER = None  # e.g. "MANUAL" to process only that classification

# ── Create persistent dirs ────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Paths configured")
print(f"  CSV        : {CSV_PATH}")
print(f"  Data dir   : {DATA_DIR}")
print(f"  Output dir : {OUTPUT_DIR}")
print(f"  Checkpoint : {OCR_CHECKPOINT}")
print()
print("MONKEYOCR_PATH will be set in the next cell after SSD copy.")

Paths configured
  CSV        : /content/mydrive/MyDrive/RAG_OCRv1/data/data_information.csv
  Data dir   : /content/mydrive/MyDrive/RAG_OCRv1/data
  Output dir : /content/mydrive/MyDrive/RAG_OCRv1/output
  Checkpoint : /content/mydrive/MyDrive/RAG_OCRv1/checkpoint/ocr_checkpoint.json

MONKEYOCR_PATH will be set in the next cell after SSD copy.


In [5]:
import shutil
import time
from pathlib import Path
from tqdm import tqdm

MONKEYOCR_DRIVE = DRIVE_BASE / "MonkeyOCR"   # source: Drive
LOCAL_BASE      = Path("/content/monkeyocr_local")
LOCAL_REPO      = LOCAL_BASE / "MonkeyOCR"    # destination: VM SSD
LOCAL_WEIGHTS   = LOCAL_REPO / "weights"


def copy_with_progress(src: Path, dst: Path):
    """Recursively copy src -> dst with a tqdm file-count progress bar."""
    all_files   = [p for p in src.rglob("*") if p.is_file()]
    total_bytes = sum(p.stat().st_size for p in all_files)

    print(f"  Files to copy : {len(all_files):,}")
    print(f"  Total size    : {total_bytes / 1024**3:.2f} GB\n")

    dst.mkdir(parents=True, exist_ok=True)

    bar          = tqdm(all_files, unit="file", desc="  Copying", dynamic_ncols=True)
    bytes_copied = 0

    for src_file in bar:
        dst_file = dst / src_file.relative_to(src)
        dst_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_file, dst_file)
        bytes_copied += src_file.stat().st_size
        bar.set_postfix({
            "copied": f"{bytes_copied / 1024**3:.2f} GB",
            "file"  : src_file.name[:35],
        })

    bar.close()


if not LOCAL_WEIGHTS.exists():
    print("Copying MonkeyOCR repo + weights: Drive -> VM local SSD")
    print("(runs once per Colab session — weights load from SSD for the rest of the session)\n")
    t0 = time.time()
    copy_with_progress(MONKEYOCR_DRIVE, LOCAL_REPO)
    elapsed = time.time() - t0
    print(f"\nCopy complete in {elapsed:.0f}s")
else:
    n_files = len(list(LOCAL_WEIGHTS.rglob("*")))
    print(f"Local copy already exists — skipping ({n_files:,} files in weights/)")

# Set MONKEYOCR_PATH to the local SSD copy
MONKEYOCR_PATH = LOCAL_REPO

print(f"\n  Local repo     : {LOCAL_REPO}")
print(f"  Local weights  : {LOCAL_WEIGHTS}")
print(f"  MONKEYOCR_PATH : {MONKEYOCR_PATH}")

Copying MonkeyOCR repo + weights: Drive -> VM local SSD
(runs once per Colab session — weights load from SSD for the rest of the session)

  Files to copy : 254
  Total size    : 7.98 GB



  Copying: 100%|██████████| 254/254 [07:34<00:00,  1.79s/file, copied=7.98 GB, file=inference.yml]


Copy complete in 467s

  Local repo     : /content/monkeyocr_local/MonkeyOCR
  Local weights  : /content/monkeyocr_local/MonkeyOCR/weights
  MONKEYOCR_PATH : /content/monkeyocr_local/MonkeyOCR


In [6]:
# 2️⃣ Paths for repo and weights
import os

BASE_DIR    = "/content/MyDrive/MyDrive/RAG_OCRv1"  # your desktop G:\My Drive\RAG_OCRv1 maps here
REPO_PATH   = os.path.join(BASE_DIR, "MonkeyOCR")
WEIGHTS_DIR = os.path.join(BASE_DIR, "MonkeyOCR_weights")

# Make sure base folder exists first
os.makedirs(BASE_DIR, exist_ok=True)
# Make weights folder
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# 3️⃣ Clone MonkeyOCR repo (if not already)
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Yuliang-Liu/MonkeyOCR.git "{REPO_PATH}"
else:
    print("MonkeyOCR repo already exists in Drive.")

Cloning into '/content/MyDrive/MyDrive/RAG_OCRv1/MonkeyOCR'...
remote: Enumerating objects: 1244, done.
remote: Counting objects: 100% (578/578), done.
remote: Compressing objects: 100% (184/184), done.
remote: Total 1244 (delta 472), reused 394 (delta 394), pack-reused 666 (from 2)
Receiving objects: 100% (1244/1244), 18.48 MiB | 18.96 MiB/s, done.
Resolving deltas: 100% (758/758), done.


## 4. Checkpoint helpers

In [7]:
import json
from datetime import datetime


def _load_checkpoint() -> dict:
    if OCR_CHECKPOINT.exists():
        try:
            with open(OCR_CHECKPOINT) as f:
                data = json.load(f)
            total_pages = sum(
                len(v.get("completed_pages", {}))
                for v in data.get("files", {}).values()
            )
            print(f"Loaded checkpoint: {len(data['files'])} file(s), "
                  f"{total_pages} page(s) already done")
            return data
        except Exception as e:
            print(f"Could not load checkpoint: {e} -- starting fresh")
    return {"last_updated": None, "files": {}}


def _save_checkpoint(data: dict):
    data["last_updated"] = datetime.now().isoformat()
    with open(OCR_CHECKPOINT, "w") as f:
        json.dump(data, f, indent=2)


def _file_key(document_group_id: str, classification: str, filename_stem: str) -> str:
    return f"{document_group_id}|{classification}|{filename_stem}"


def _is_page_done(cp: dict, fkey: str, page_num: int) -> bool:
    return str(page_num) in cp["files"].get(fkey, {}).get("completed_pages", {})


def _init_file_entry(cp: dict, fkey: str, total_pages: int):
    if fkey not in cp["files"]:
        cp["files"][fkey] = {
            "status":          "in_progress",
            "total_pages":     total_pages,
            "completed_pages": {},
            "failed_pages":    {},
            "started_at":      datetime.now().isoformat(),
            "completed_at":    None,
        }
    else:
        cp["files"][fkey]["total_pages"] = total_pages
    _save_checkpoint(cp)


def _mark_page_done(cp: dict, fkey: str, page_num: int, output_page_dir: Path):
    cp["files"][fkey]["completed_pages"][str(page_num)] = {
        "output_dir": str(output_page_dir),
        "timestamp":  datetime.now().isoformat(),
    }
    _save_checkpoint(cp)


def _mark_page_failed(cp: dict, fkey: str, page_num: int, error: str):
    cp["files"][fkey]["failed_pages"][str(page_num)] = {
        "error":     error,
        "timestamp": datetime.now().isoformat(),
    }
    _save_checkpoint(cp)


def _mark_file_complete(cp: dict, fkey: str, item: dict,
                        total_pages: int, output_file_dir: Path):
    entry = cp["files"].get(fkey, {})
    entry.update({
        "status":            "completed",
        "total_pages":       total_pages,
        "completed_at":      datetime.now().isoformat(),
        "output_dir":        str(output_file_dir),
        "document_group_id": item.get("document_group_id", ""),
        "classification":    item.get("classification", ""),
        "filename":          item.get("filename", ""),
        # Optional CSV fields — only written when non-empty
        **{k: item[k] for k in
           ("category_level_1", "category_level_2",
            "model_number", "date_added", "notes")
           if item.get(k)},
    })
    cp["files"][fkey] = entry
    _save_checkpoint(cp)


print("Checkpoint helpers defined")

Checkpoint helpers defined


## 5. Load CSV manifest

In [8]:
import pandas as pd

OPTIONAL_COLS = [
    "category_level_1", "category_level_2",
    "model_number", "date_added", "notes",
]

df = pd.read_csv(CSV_PATH)

required = {"document_group_id", "filename", "classification"}
missing  = required - set(df.columns)
if missing:
    raise ValueError(f"CSV is missing required columns: {missing}")

df["classification"] = df["classification"].str.strip().str.upper()

print(f"Loaded {len(df)} row(s) from {CSV_PATH.name}")
print(f"  Document groups : {df['document_group_id'].nunique()}")
print(f"  Classifications : {sorted(df['classification'].unique())}")
df.head()

Loaded 8 row(s) from data_information.csv
  Document groups : 8
  Classifications : ['MANUAL']


,document_group_id,filename,classification,category_level_1,category_level_2,model_number,date_added,notes
0,panasonic_aircon_CS-PW24KE,CS-PW24KE_service_manual_PHAAM0810057C2.pdf,MANUAL,Air_Conditioner,NaN,CS-PW24KE,31/3/2026,NaN
1,panasonic_aircon_CS-E12QD3EAW,CS-E12QD3EAW_SM_PAPAMY1406071CE.pdf,MANUAL,Air_Conditioner,NaN,"CS-E12QD3EAW,CU-E12QD3EA",31/3/2026,NaN
2,panasonic_aircon_E7JKEW,CS-E7JKEW_SM_PHAAM0810051C2.pdf,MANUAL,Air_Conditioner,NaN,"CS-E7JKEW,CU-E7JKE,CS-E9JKEW,CU-E9JKE,CS-E12JK...",8/4/2026,NaN
3,panasonic_aircon_CS-C18DKV,CS-C18DKV.pdf,MANUAL,Air_Conditioner,NaN,"CS-C18DKV,CU-C18DKV,CS-C24DKV,CU-C24DKV",8/4/2026,NaN
4,toshiba_aircon_RAS-M22E2KV2G-A,RAS-M22E2KV2G-A RAS-M24E2KV2G-A-1696909076.pdf,MANUAL,Air_Conditioner,NaN,"RAS-M22E2KV2G-A,RAS-M24E2KV2G-A",8/4/2026,NaN


## 6. Build work list from CSV

In [9]:
work_items = []

for _, row in df.iterrows():
    doc_group_id   = str(row["document_group_id"]).strip()
    filename       = str(row["filename"]).strip()
    classification = str(row["classification"]).strip().upper()

    if DOC_GROUP_FILTER and doc_group_id != DOC_GROUP_FILTER:
        continue
    if CLASSIFICATION_FILTER and classification != CLASSIFICATION_FILTER.upper():
        continue

    pdf_path = DATA_DIR / doc_group_id / filename
    if not pdf_path.exists():
        print(f"  File not found, skipping: {pdf_path}")
        continue

    optional_meta = {
        col: str(row[col]).strip()
        for col in OPTIONAL_COLS
        if col in df.columns
        and pd.notna(row.get(col))
        and str(row.get(col)).strip()
    }

    work_items.append({
        "document_group_id": doc_group_id,
        "filename":          filename,
        "filename_stem":     Path(filename).stem,
        "classification":    classification,
        "pdf_path":          pdf_path,
        **optional_meta,
    })

if LIMIT:
    seen, limited = set(), []
    for item in work_items:
        seen.add(item["document_group_id"])
        limited.append(item)
        if len(seen) >= LIMIT:
            break
    work_items = limited

print(f"Work list: {len(work_items)} PDF file(s)")
for item in work_items:
    print(f"  [{item['classification']}] {item['document_group_id']} / {item['filename']}")

Work list: 8 PDF file(s)
  [MANUAL] panasonic_aircon_CS-PW24KE / CS-PW24KE_service_manual_PHAAM0810057C2.pdf
  [MANUAL] panasonic_aircon_CS-E12QD3EAW / CS-E12QD3EAW_SM_PAPAMY1406071CE.pdf
  [MANUAL] panasonic_aircon_E7JKEW / CS-E7JKEW_SM_PHAAM0810051C2.pdf
  [MANUAL] panasonic_aircon_CS-C18DKV / CS-C18DKV.pdf
  [MANUAL] toshiba_aircon_RAS-M22E2KV2G-A / RAS-M22E2KV2G-A RAS-M24E2KV2G-A-1696909076.pdf
  [MANUAL] toshiba_aircon_RAS-M10SKV-E / RAS-M16SKV-E-1434518040.pdf
  [MANUAL] toshiba_aircon_RAS-30-BKVS-A / RAS-30,34,36BAVS-A-series.pdf
  [MANUAL] toshiba_aircon_RAS-22SKV2-A1 / RAS-22SKV2-A1_and_RAS-22SAV2-A1-1434509223.pdf


## 7. OCR — page-by-page with checkpointing

For each PDF:
- Convert one page at a time to PNG at 300 DPI (memory-efficient)
- Run MonkeyOCR on the page image
- Save outputs to `output/{classification}/{document_group_id}/{filename}/page_{N}/`
- Mark the page done in the checkpoint; already-done pages are skipped

In [ ]:
import subprocess
import shutil
import pymupdf as fitz

checkpoint = _load_checkpoint()

ZOOM = DPI / 72  # fitz zoom factor for target DPI

# ── Summary counters ──────────────────────────────────────────────────────────
total_files_done   = 0
total_files_skip   = 0
total_pages_done   = 0
total_pages_skip   = 0
total_pages_failed = 0

print("=" * 10)
print("PHASE 1 — OCR")
print("=" * 10)

for item in work_items:
    doc_group_id   = item["document_group_id"]
    filename       = item["filename"]
    filename_stem  = item["filename_stem"]
    classification = item["classification"]
    pdf_path       = item["pdf_path"]

    fkey = _file_key(doc_group_id, classification, filename_stem)

    # Output directory for this file:
    # output/{classification}/{document_group_id}/{filename}/
    file_out_dir = OUTPUT_DIR / classification / doc_group_id / filename_stem
    file_out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"📦 {doc_group_id} / {filename}")
    print(f"   Classification : {classification}")
    print(f"   PDF path       : {pdf_path}")
    print(f"   Output dir     : {file_out_dir}")
    print(f"{'='*70}")

    # Check if already fully completed
    file_entry = checkpoint["files"].get(fkey, {})
    if file_entry.get("status") == "completed":
        print(f"   ✅ SKIP — fully completed at {file_entry.get('completed_at', '?')}")
        total_files_skip += 1
        continue

    # Open PDF (only for page count + per-page rendering)
    try:
        doc = fitz.open(str(pdf_path))
    except Exception as e:
        print(f"   ❌ Cannot open PDF: {e}")
        continue

    total_pages = len(doc)
    print(f"   Total pages : {total_pages}")

    # Initialise file entry in checkpoint if not present
    if fkey not in checkpoint["files"]:
        checkpoint["files"][fkey] = {
            "status":          "in_progress",
            "total_pages":     total_pages,
            "completed_pages": {},
            "failed_pages":    {},
            "started_at":      datetime.now().isoformat(),
            "completed_at":    None,
        }
        _save_checkpoint(checkpoint)
    else:
        # Update total_pages in case it was unknown
        checkpoint["files"][fkey]["total_pages"] = total_pages

    file_pages_done   = 0
    file_pages_skip   = 0
    file_pages_failed = 0

    for page_idx in range(total_pages):
        page_num = page_idx + 1  # 1-indexed

        # ── Skip if already done ──────────────────────────────────────────
        if _is_page_done(checkpoint, fkey, page_num):
            print(f"   page {page_num:>4}/{total_pages} ✅ SKIP (cached)")
            file_pages_skip  += 1
            total_pages_skip += 1
            continue

        print(f"\n   page {page_num:>4}/{total_pages} — processing...")

        # ── Output dir for this page ──────────────────────────────────────
        page_out_dir = file_out_dir / f"page_{page_num}"
        page_out_dir.mkdir(parents=True, exist_ok=True)

        # ── Step A: Rasterise single page → page.png ──────────────────────
        try:
            page_obj = doc.load_page(page_idx)
            mat      = fitz.Matrix(ZOOM, ZOOM)
            pix      = page_obj.get_pixmap(matrix=mat, alpha=False)
            page_png = page_out_dir / "page.png"
            pix.save(str(page_png))
            del pix, page_obj  # free memory before OCR
            print(f"            🖼  Saved: {page_png.name}")
        except Exception as e:
            err = f"Rasterise failed: {e}"
            print(f"            ❌ {err}")
            _mark_page_failed(checkpoint, fkey, page_num, err)
            file_pages_failed  += 1
            total_pages_failed += 1
            continue

        # ── Step B: MonkeyOCR ─────────────────────────────────────────────
        # MonkeyOCR writes its output into a subdirectory inside page_out_dir
        temp_ocr_dir = page_out_dir / "_temp_ocr"
        temp_ocr_dir.mkdir(parents=True, exist_ok=True)

        try:
            result = subprocess.run(
                [
                    "python", "parse.py",
                    str(page_png),
                    "-o", str(temp_ocr_dir),
                ],
                cwd=str(MONKEYOCR_PATH),
                capture_output=True,
                text=True,
            )

            if result.returncode != 0:
                raise RuntimeError(result.stderr[:500])

            # ── Move MonkeyOCR outputs up into page_out_dir ───────────────
            # MonkeyOCR may nest outputs one level deeper (named after the image)
            nested = list(temp_ocr_dir.iterdir())
            if len(nested) == 1 and nested[0].is_dir():
                src_dir = nested[0]
            else:
                src_dir = temp_ocr_dir

            for item_path in src_dir.iterdir():
                dest = page_out_dir / item_path.name
                if dest.exists():
                    if dest.is_dir():
                        shutil.rmtree(dest)
                    else:
                        dest.unlink()
                shutil.move(str(item_path), str(dest))

            # Clean up temp dir
            shutil.rmtree(temp_ocr_dir, ignore_errors=True)

            # Quick preview: first non-empty line of markdown output
            md_files = list(page_out_dir.glob("*.md"))
            if md_files:
                preview = ""
                for line in md_files[0].read_text(encoding="utf-8").splitlines():
                    if line.strip():
                        preview = line.strip()[:100]
                        break
                print(f"            📝 Preview: {preview}")

            _mark_page_done(checkpoint, fkey, page_num, page_out_dir)
            print(f"            ✅ Page {page_num} complete → {page_out_dir.name}/")
            file_pages_done   += 1
            total_pages_done  += 1

        except Exception as e:
            err = str(e)[:300]
            print(f"            ❌ MonkeyOCR failed: {err}")
            shutil.rmtree(temp_ocr_dir, ignore_errors=True)
            _mark_page_failed(checkpoint, fkey, page_num, err)
            file_pages_failed  += 1
            total_pages_failed += 1

    doc.close()

    # ── Mark file complete if all pages succeeded ─────────────────────────
    done_count   = len(checkpoint["files"][fkey]["completed_pages"])
    failed_count = len(checkpoint["files"][fkey]["failed_pages"])

    if done_count == total_pages:
        _mark_file_complete(checkpoint, fkey, item, total_pages, file_out_dir)
        print(f"\n   🏁 File complete: {done_count}/{total_pages} pages")
        total_files_done += 1
    else:
        # Partial — update counters but keep status as in_progress
        checkpoint["files"][fkey]["total_pages"] = total_pages
        _save_checkpoint(checkpoint)
        print(f"\n   ⚠️  Partial: {done_count} done, {failed_count} failed / {total_pages} total")

    print(f"   Pages this file — done: {file_pages_done}  "
          f"skip: {file_pages_skip}  failed: {file_pages_failed}")

print("\n" + "=" * 70)
print("🏁 PHASE 1 OCR — COMPLETE")
print("=" * 70)
print(f"  Files completed  : {total_files_done}")
print(f"  Files skipped    : {total_files_skip}")
print(f"  Pages OCR done   : {total_pages_done}")
print(f"  Pages skipped    : {total_pages_skip}")
print(f"  Pages failed     : {total_pages_failed}")
print(f"  Checkpoint saved : {OCR_CHECKPOINT}")
print("=" * 70)

Loaded checkpoint: 3 file(s), 262 page(s) already done
PHASE 1 — OCR

📦 panasonic_aircon_CS-PW24KE / CS-PW24KE_service_manual_PHAAM0810057C2.pdf
   Classification : MANUAL
   PDF path       : /content/mydrive/MyDrive/RAG_OCRv1/data/panasonic_aircon_CS-PW24KE/CS-PW24KE_service_manual_PHAAM0810057C2.pdf
   Output dir     : /content/mydrive/MyDrive/RAG_OCRv1/output/MANUAL/panasonic_aircon_CS-PW24KE/CS-PW24KE_service_manual_PHAAM0810057C2
   ✅ SKIP — fully completed at 2026-04-05T05:39:22.188598

📦 panasonic_aircon_CS-E12QD3EAW / CS-E12QD3EAW_SM_PAPAMY1406071CE.pdf
   Classification : MANUAL
   PDF path       : /content/mydrive/MyDrive/RAG_OCRv1/data/panasonic_aircon_CS-E12QD3EAW/CS-E12QD3EAW_SM_PAPAMY1406071CE.pdf
   Output dir     : /content/mydrive/MyDrive/RAG_OCRv1/output/MANUAL/panasonic_aircon_CS-E12QD3EAW/CS-E12QD3EAW_SM_PAPAMY1406071CE
   ✅ SKIP — fully completed at 2026-04-08T12:12:25.342760

📦 panasonic_aircon_E7JKEW / CS-E7JKEW_SM_PHAAM0810051C2.pdf
   Classification : MANUAL
  

In [10]:
!pip install PyMuPDF==1.27.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 70.4 MB/s eta 0:00:00
  Attempting uninstall: PyMuPDF
    Found existing installation: PyMuPDF 1.24.14
    Uninstalling PyMuPDF-1.24.14:
      Successfully uninstalled PyMuPDF-1.24.14


## 8. Checkpoint inspection (optional)

Run this cell any time to see what has been processed so far.

In [ ]:
import json
from pathlib import Path

if OCR_CHECKPOINT.exists():
    cp = json.loads(OCR_CHECKPOINT.read_text())
    print(f"Last updated : {cp.get('last_updated')}")
    print(f"Total files  : {len(cp['files'])}\n")

    for key, entry in cp["files"].items():
        doc_group_id, classification, filename_stem = key.split("|", 2)
        n_done   = len(entry.get("completed_pages", {}))
        n_failed = len(entry.get("failed_pages", {}))
        total    = entry.get("total_pages", "?")
        status   = entry.get("status", "unknown")
        icon     = "✅" if status == "completed" else ("❌" if n_failed else "⏳")
        print(f"  {icon} [{classification}] {doc_group_id} / {filename_stem}")
        print(f"      pages: {n_done}/{total} done, {n_failed} failed  |  status: {status}")
        if n_failed:
            for pg, info in entry["failed_pages"].items():
                print(f"      ⚠️  page {pg}: {info['error'][:80]}")
else:
    print("No checkpoint found yet.")

## 9. Legecy code for MonkeyOCR validation (No need to run)

Run this cell any time to see what has been processed so far.

In [ ]:
import subprocess
from pathlib import Path
from google.colab import drive
import pymupdf as fitz



# 2️⃣ Paths
DRIVE_BASE = Path("/content/mydrive/MyDrive/RAG_OCRv1")
MONKEY_OCR_PATH = DRIVE_BASE / "MonkeyOCR"
pdf_path = DRIVE_BASE / "data/panasonic_aircon_CS-PW24KE/CS-PW24KE_service_manual_PHAAM0810057C2.pdf"
images_dir = DRIVE_BASE / "temp_images"
output_dir = DRIVE_BASE / "output"
images_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

# 3️⃣ Open PDF
print("📄 Processing PDF page by page...")

doc = fitz.open(str(pdf_path))
zoom = 300 / 72  # for 300 DPI

# 4️⃣ Loop: convert → OCR → repeat
for i, page in enumerate(doc):
    print(f"\n📄 Page {i+1}")

    # 🔹 Convert ONE page to image
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img_file = images_dir / f"page_{i+1}.png"
    pix.save(str(img_file))

    print(f"🖼 Converted: {img_file.name}")

    # 🔹 Run OCR immediately
    print(f"🔍 Running OCR on page {i+1}...")

    result = subprocess.run(
        [
            "python", "parse.py",
            str(img_file),
            "-o", str(output_dir)
        ],
        cwd=str(MONKEY_OCR_PATH),
        capture_output=True,
        text=True
    )

    print(result.stdout)

    if result.returncode != 0:
        print("❌ Error:")
        print(result.stderr)
        break  # optional: stop if one page fails

print(f"\n✅ Done! Output saved to: {output_dir}")

📄 Processing PDF page by page...

📄 Page 1
🖼 Converted: page_1.png
🔍 Running OCR on page 1...
Loading model...
Starting to parse file: /content/mydrive/MyDrive/RAG_OCRv1/temp_images/page_1.png
Output dir: /content/mydrive/MyDrive/RAG_OCRv1/output/page_1
Performing document parsing...
Processing as single result...
Parsing and saving time: 86.82s
Results saved to  /content/mydrive/MyDrive/RAG_OCRv1/output/page_1

✅ Parsing completed! Results saved in: /content/mydrive/MyDrive/RAG_OCRv1/output/page_1


📄 Page 2
🖼 Converted: page_2.png
🔍 Running OCR on page 2...


KeyboardInterrupt: 

In [ ]:
# # 1️⃣ Generate requirements.txt with all installed packages
# !pip freeze > requirements.txt

# # 2️⃣ View the contents
# !cat requirements.txt

absl-py==1.4.0
accelerate==1.13.0
access==1.1.10.post3
acres==0.5.0
addict==2.4.0
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.4
aiosignal==1.4.0
aiosqlite==0.22.1
aistudio-sdk==0.3.8
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.4
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.13.0
anywidget==0.9.21
apsw==3.51.3.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.3.30.0.54.34
astunparse==1.6.3
atpublic==5.1
attrs==26.1.0
audioread==3.1.0
Authlib==1.6.9
autograd==1.8.0
babel==2.18.0
backcall==0.2.0
bce-python-sdk==0.9.68
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.38.0
bigquery-magics==0.12.2
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.2.0
blosc2==4.1.2
bokeh==3.8.2
boto3==1.42.83
botocore==1.42.83
Bottleneck==1.4.2
bqplot==0.12.45
br